In [2]:
# Imports

from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Set up paths

project_root = Path.cwd().parent

metadata_path = project_root / 'data' / 'preprocessed' / 'metadata.csv'
landmark_dir = project_root / 'data' / 'preprocessed' / 'landmarks'

In [4]:
# Loading processed metadata file as pandas dataframe
metadata = pd.read_csv(metadata_path)
metadata.head()

,video_id,gloss,signer_id,split,fps,frame_count,frames_with_hands
0,69302,drink,115,val,29.970030,77,54
1,65539,drink,94,train,23.976169,44,17
2,17710,drink,36,train,29.970000,70,32
3,17733,drink,12,train,30.296090,93,46
4,65540,drink,94,train,23.976093,46,24


In [5]:
# Inspecting sequence lengths
metadata["frame_count"].describe()

count    345.000000
mean      63.826087
std       26.509847
min       20.000000
25%       43.000000
50%       60.000000
75%       84.000000
max      195.000000
Name: frame_count, dtype: float64

In [6]:
# Unique split counts
metadata["split"].value_counts()

split
train    241
val       61
test      43
Name: count, dtype: int64

In [7]:
# ASL word by count
metadata["gloss"].value_counts().sort_index()

gloss
accident        13
basketball      12
before          16
bowling         13
call            12
candy           13
change          12
cold            12
computer        14
cool            16
cousin          14
dark            12
drink           15
go              15
help            14
last            12
pizza           12
shirt           12
short           13
tall            13
thanksgiving    13
thin            16
trade           13
what            12
who             14
yes             12
Name: count, dtype: int64

In [8]:
# Inspecting a single landmark array

sample_path = landmark_dir / '69302.npy'

sample = np.load(sample_path)

print(f"Shape: {sample.shape}")
print(f"Data type: {sample.dtype}")
print(f"First frame:\n {sample[0]}")
print(f"Minimum: {sample.min()}")
print(f"Maximum: {sample.max()}")

Shape: (77, 2, 21, 3)
Data type: float32
First frame:
 [[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]
Minimum: -0.05317000299692154
Maximum: 1.0550733804702759


In [9]:
# Identify the landmarks that were actually detected
valid_mask = np.any(sample != 0, axis = -1)
valid_landmarks = sample[valid_mask]

print("Valid landmarks: ", len(valid_landmarks))

print(f"X range: {valid_landmarks[:, 0].min()}, {valid_landmarks[:, 0].max()}")
print(f"Y range: {valid_landmarks[:, 1].min()}, {valid_landmarks[:, 1].max()}")
print(f"Z range: {valid_landmarks[:, 2].min()}, {valid_landmarks[:, 2].max()}")

Valid landmarks:  1134
X range: 0.4249601662158966, 0.5629324913024902
Y range: 0.28702211380004883, 1.0550733804702759
Z range: -0.05317000299692154, 0.005302421282976866


In [10]:
# Number of frames where each hand slot contains detected landmarks
hand_presence = np.any(sample != 0, axis = (2, 3))

print(f"Frames with left-slot landmarks: {hand_presence[:, 0].sum()}")
print(f"Frames with right-slot landmarks: {hand_presence[:, 1].sum()}")
print(f"Frames with neither hand detected: {(~hand_presence.any(axis=1)).sum()}")

Frames with left-slot landmarks: 0
Frames with right-slot landmarks: 54
Frames with neither hand detected: 23


In [11]:
# Values we'll use for preparing batches

lengths = metadata["frame_count"]
print(f"Minimum: {lengths.min()}")
print(f"Maximum: {lengths.max()}")
print(f"Mean: {lengths.mean()}")
print(f"Median: {lengths.median()}")

Minimum: 20
Maximum: 195
Mean: 63.82608695652174
Median: 60.0


In [12]:
# Label mapping

labels = sorted(metadata["gloss"].unique())

label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print(f"Number of classes: {len(labels)}")
print(label_to_id)

Number of classes: 26
{'accident': 0, 'basketball': 1, 'before': 2, 'bowling': 3, 'call': 4, 'candy': 5, 'change': 6, 'cold': 7, 'computer': 8, 'cool': 9, 'cousin': 10, 'dark': 11, 'drink': 12, 'go': 13, 'help': 14, 'last': 15, 'pizza': 16, 'shirt': 17, 'short': 18, 'tall': 19, 'thanksgiving': 20, 'thin': 21, 'trade': 22, 'what': 23, 'who': 24, 'yes': 25}


# CREATING PYTORCH DATASET

In [13]:
# Imports
import torch
from torch.utils.data import Dataset

In [30]:
class ASLDataset(Dataset):

    def __init__(self, metadata, landmark_dir, label_to_id):
        self.metadata = metadata.reset_index(drop=True)
        self.landmark_dir = landmark_dir
        self.label_to_id = label_to_id

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        landmark_path = self.landmark_dir / f"{int(row["video_id"]):05d}.npy"
        landmarks = np.load(landmark_path)

        features = landmarks.reshape(landmarks.shape[0], -1)
        label = self.label_to_id[row["gloss"]]

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )

In [31]:
# Split data into three official splits
train_metadata = metadata[metadata["split"] == "train"]
val_metadata = metadata[metadata["split"] == "val"]
test_metadata = metadata[metadata["split"] == "test"]

train_dataset = ASLDataset(train_metadata, landmark_dir, label_to_id)
val_dataset = ASLDataset(val_metadata, landmark_dir, label_to_id)
test_dataset = ASLDataset(test_metadata, landmark_dir, label_to_id)

print("Train: ", len(train_dataset))
print("Val: ", len(val_dataset))
print("Test: ", len(test_dataset))

Train:  241
Val:  61
Test:  43


In [32]:
# Retrieving one sample
features, label = train_dataset[0]

print("Features shape: ", features.shape)
print("Features dtype: ", features.dtype)
print("Label: ", label)
print("Label dtype: ", label.dtype)
print("Word: ", id_to_label[label.item()])

Features shape:  torch.Size([44, 126])
Features dtype:  torch.float32
Label:  tensor(12)
Label dtype:  torch.int64
Word:  drink


# Creating a padding function

In [33]:
# Imports
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [34]:
def collate_fn(batch):
    sequences, labels = zip(*batch)     # separates video sequences and their labels

    # record each video's original frame count before padding
    lengths = torch.tensor([len(seq) for seq in sequences])

    # Adds zeros to shorter videos
    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,   # makes resulting shape (batch_size, max_frames, 126)
        padding_value=0.0,
    )

    labels = torch.stack(labels)    # combines individual labels into one tensor

    return padded_sequences, lengths, labels

# Create the DataLoaders

In [35]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,  # 16 videos at a time
    shuffle=True,   # randomize training examples each epoch
    collate_fn=collate_fn,  # tells dataloader how to combine videos of different lengths
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
)

# Inspect one batch

In [39]:
videos, lengths, labels = next(iter(train_loader))

print("Videos: ", videos.shape)
print("Lengths: ", lengths.shape)
print("Labels: ", labels.shape)

Videos:  torch.Size([16, 155, 126])
Lengths:  torch.Size([16])
Labels:  torch.Size([16])


# Build the LSTM model

In [40]:
# Imports
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

In [41]:
# Create the model
# LSTM processes each video's sequence of hand landmarks and learns movement patterns over time
# Hidden state captures info learned from the sequence
# Fully connected layer converts that info into 26 class scores
# Outputs one score per ASL word
class ASLClassifier(nn.Module):

    def __init__(self, input_size=126, hidden_size=128, num_classes=26):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, videos, lengths):
        packed = pack_padded_sequence(
            videos,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, _) = self.lstm(packed)

        output = self.fc(hidden[-1])

        return output

In [42]:
# Test the model

model = ASLClassifier()
videos, lengths, labels = next(iter(train_loader))
outputs = model(videos, lengths)
print("Output shape: ", outputs.shape)

Output shape:  torch.Size([16, 26])


In [43]:
# Set up the loss function and optimizer
criterion = nn.CrossEntropyLoss()   # how far model's pred from actual labels

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001,
)

In [44]:
# Select the device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print("Using device: ", device)

Using device:  mps


In [45]:
# Training loop
num_epochs = 20     # Number of epochs

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for videos, lengths, labels in train_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos, lengths)
        loss = criterion(outputs, labels)

        loss.backward()     # calculate gradients
        optimizer.step()    # update weights

        total_loss+=loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{num_epochs}] Loss: {avg_loss:.4f}]")

Epoch [1/20] Loss: 3.2699]
Epoch [2/20] Loss: 3.2344]
Epoch [3/20] Loss: 3.1415]
Epoch [4/20] Loss: 3.1271]
Epoch [5/20] Loss: 3.0668]
Epoch [6/20] Loss: 3.3501]
Epoch [7/20] Loss: 3.2376]
Epoch [8/20] Loss: 3.1964]
Epoch [9/20] Loss: 3.1296]
Epoch [10/20] Loss: 3.0273]
Epoch [11/20] Loss: 3.0114]
Epoch [12/20] Loss: 2.9686]
Epoch [13/20] Loss: 2.9179]
Epoch [14/20] Loss: 2.9401]
Epoch [15/20] Loss: 2.9297]
Epoch [16/20] Loss: 2.8476]
Epoch [17/20] Loss: 2.8792]
Epoch [18/20] Loss: 2.8159]
Epoch [19/20] Loss: 2.8077]
Epoch [20/20] Loss: 2.7561]


In [46]:
# Evaluate on the validation set
model.eval()    # switch to evaluation mode

val_loss = 0
correct = 0
total = 0

with torch.no_grad():   # prevents gradient calculation
    for videos, lengths, labels in val_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos, lengths)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim = 1)   # select class with highest score for each video

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

avg_val_loss = val_loss / total
val_accuracy = 100 * correct / total

print(f"Validation loss: {avg_val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2f}")

Validation loss: 2.9648
Validation accuracy: 13.11


In [47]:
# Inspecting model's predictions

model.eval()    # switch to evaluation mode

videos, lengths, labels = next(iter(val_loader))

with torch.no_grad():
    outputs = model(videos.to(device), lengths)
    predictions = outputs.argmax(dim=1).cpu()

for i in range(len(labels)):
    actual = id_to_label[labels[i].item()]
    predicted = id_to_label[predictions[i].item()]

    print(f"Actual: {actual:15} Predicted: {predicted}")

Actual: drink           Predicted: who
Actual: drink           Predicted: candy
Actual: drink           Predicted: drink
Actual: computer        Predicted: trade
Actual: computer        Predicted: change
Actual: computer        Predicted: trade
Actual: before          Predicted: basketball
Actual: go              Predicted: basketball
Actual: go              Predicted: change
Actual: go              Predicted: drink
Actual: who             Predicted: who
Actual: who             Predicted: who
Actual: who             Predicted: drink
Actual: candy           Predicted: candy
Actual: candy           Predicted: bowling
Actual: cousin          Predicted: cousin
